In [1]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # 解决你之前的报错

import shap
import torch
from transformers import pipeline

# 1. 加载和你之前脚本一样的模型
model_name = "roberta-large-mnli"
# 这里直接用 pipeline，SHAP 对其有原生支持
classifier = pipeline("text-classification", model=model_name, top_k=None)

# 2. 定义解释器
# SHAP 会自动识别这是一个文本序列任务
explainer = shap.Explainer(classifier)

# 3. 准备要分析的例子 (从你的 train_nli.jsonl 中选几个代表性的)
# 格式必须是模型接受的字符串格式
test_texts = [
    "This is a bread knife. </s> </s> The knife is used for the purpose of bread.",
    "This is a steel table. </s> </s> The table contains or holds the steel."
]

# 4. 计算 SHAP 值
# 这步可能比较慢，因为模型需要反复扰动输入词来计算贡献度
shap_values = explainer(test_texts)

# 5. 可视化
# 在 Jupyter Notebook 中运行此行可以直接看到彩色交互图
# 如果在脚本中运行，可以保存为 html
shap.plots.text(shap_values[0, :, "ENTAILMENT"])

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/462 [00:00<?, ?it/s]

PartitionExplainer explainer:  50%|█████     | 1/2 [00:00<?, ?it/s]

  0%|          | 0/380 [00:00<?, ?it/s]

PartitionExplainer explainer: 3it [00:35, 17.67s/it]               


In [3]:
# 抽取预测错误的样本
import os
import json
import shap
from transformers import pipeline

# 解决 Mac 或多环境下的 OpenMP 冲突报错
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def analyze_errors_with_shap(results_file="nli_results_full.jsonl", top_n=3):
    # 1. 读取并筛选错误样本
    print("正在读取评估结果并筛选错误样本...")
    error_samples = []
    
    with open(results_file, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # 我们的假设(Hypothesis)本来就应该被推导出来，所以预期是 ENTAILMENT
            # 如果模型预测的不是 ENTAILMENT，那就是预测错了
            if data.get('predicted_nli') != 'ENTAILMENT':
                error_samples.append(data)

    if not error_samples:
        print("太棒了！没有找到任何预测错误的样本。")
        return

    # 按置信度降序排序，找出模型“最自信但却大错特错”的样本
    error_samples.sort(key=lambda x: x.get('conf_score', 0), reverse=True)
    target_samples = error_samples[:top_n]
    
    print(f"找到 {len(error_samples)} 个错误样本。将对最典型的 {len(target_samples)} 个运行 SHAP 分析。")

    # 2. 准备 SHAP 解释器
    print("正在加载 RoBERTa-large-mnli 模型 (这可能需要一点时间)...")
    # 注意：top_k=None 会返回所有类别的概率，这是 SHAP 计算所必需的
    classifier = pipeline("text-classification", model="roberta-large-mnli", device=-1, top_k=None)
    explainer = shap.Explainer(classifier)

    # 3. 提取文本格式
    # 必须拼接成模型认识的格式：Premise </s> </s> Hypothesis
    texts_to_analyze = []
    for data in target_samples:
        text = f"{data['premise']} </s> </s> {data['hypothesis']}"
        texts_to_analyze.append(text)
        print(f"分析目标 [{data['original_label']}]: {data['compound']} (预测为 {data['predicted_nli']})")

    # 4. 计算 SHAP 值
    print("正在计算 SHAP 值 (模型正在进行词汇扰动，请耐心等待)...")
    shap_values = explainer(texts_to_analyze)

    # 5. 生成并保存可视化 HTML
    # display=False 会返回 HTML 字符串，而不是直接在终端尝试渲染
    html_content = shap.plots.text(shap_values, display=False)
    
    output_html = "shap_error_analysis.html"
    with open(output_html, "w", encoding="utf-8") as f:
        # 添加一点基础的 HTML 结构使其更好看
        f.write("<html><head><meta charset='utf-8'><title>NLI SHAP Error Analysis</title></head><body style='padding: 20px;'>")
        f.write("<h2>NLI 错误样本 SHAP 可视化报告</h2>")
        f.write("<p>红色高亮表示推动模型预测该结果的词，蓝色表示拉低该结果的词。</p>")
        f.write(html_content)
        f.write("</body></html>")
        
    print(f"\n✅ SHAP 分析完成！")
    print(f"请在浏览器中打开文件查看交互式报告：{os.path.abspath(output_html)}")

if __name__ == "__main__":
    # 确保你的目录下有上一步生成的 nli_results_full.jsonl
    analyze_errors_with_shap("nli_results_full.jsonl", top_n=3)

正在读取评估结果并筛选错误样本...
找到 632 个错误样本。将对最典型的 3 个运行 SHAP 分析。
正在加载 RoBERTa-large-mnli 模型 (这可能需要一点时间)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


分析目标 [other]: science graduate (预测为 NEUTRAL)
分析目标 [other]: crown prince (预测为 NEUTRAL)
分析目标 [other]: minority people (预测为 NEUTRAL)
正在计算 SHAP 值 (模型正在进行词汇扰动，请耐心等待)...


  0%|          | 0/498 [00:00<?, ?it/s]

...

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 3/3 [02:22<00:00, 35.32s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 4it [03:27, 69.01s/it]                       


✅ SHAP 分析完成！
请在浏览器中打开文件查看交互式报告：/Users/yanniss/Documents/Semantik/project/NLI/shap_error_analysis.html
